# Data Science: Feature Selection via Continuous Relaxation

Feature-subset selection is naturally binary (include/exclude each feature), but instead of writing a separate binary-encoded algorithm we reuse the continuous optimizers: each dimension is a weight $w_i \in [0, 1]$, thresholded at $0.5$ to get the subset mask inside the objective function. This is the same trick behind 'sigmoid'/continuous-relaxation binary PSO in the literature, and it means any of `GeneticAlgorithm`, `ParticleSwarmOptimization`, `DifferentialEvolution`, or `SlimeMouldAlgorithm` can be used unmodified.

## Why this is hard

Selecting a subset of $N$ features is a combinatorial problem: there are $2^N$ possible subsets, so for the 30 features in this dataset alone there are over a billion candidates. Exhaustive search is out, and even greedy wrapper methods (forward selection, backward elimination) require $O(N^2)$ model evaluations and can get stuck in subsets that look good locally but aren't. Framing the subset mask as a continuous vector lets population-based metaheuristics explore many candidate subsets in parallel and escape those local optima, at the cost of needing many objective evaluations — each of which, here, is itself a 5-fold cross-validation.

Feature selection methods are usually grouped into three families:

- **Filter methods** score each feature independently of any model (e.g. correlation, mutual information, chi-squared) — fast, but ignore feature interactions.
- **Wrapper methods** (what this notebook does) use the performance of a downstream model, evaluated via cross-validation, as the selection criterion — captures interactions, but is expensive because it requires training a model per candidate subset.
- **Embedded methods** perform selection as part of model fitting itself (e.g. L1-regularized linear models, tree feature importances) — a middle ground, tied to a specific model family.

The cost function this notebook actually optimizes is developed in its own section below, once the data is loaded.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

from metaheuristics.algorithms.particle_swarm import ParticleSwarmOptimization

X, y = load_breast_cancer(return_X_y=True)
NUM_FEATURES = X.shape[1]
ALPHA = 0.02  # weight on the feature-count penalty

print(f"Number of features: {NUM_FEATURES} and sample size: {X.shape[0]}")
print(f"ALPHA: {ALPHA}")
print(f"Type of X: {type(X)}")

Number of features: 30 and sample size: 569
ALPHA: 0.02
Type of X: <class 'numpy.ndarray'>


## The Cost Function

### From a discrete to a continuous problem

The natural formulation of feature selection is a binary combinatorial problem: choose an indicator vector $s \in \{0, 1\}^N$ minimizing

$$g(s) = \big(1 - \text{CV accuracy}(s)\big) + \alpha \cdot \frac{\|s\|_0}{N}, \qquad \|s\|_0 = \sum_i s_i$$

where $\|s\|_0$ (the $L_0$ "norm") counts the selected features. This is the same penalized-loss shape behind AIC/BIC-style model selection and sparse regression — except here $\|s\|_0$ is used *directly*, not approximated by a convex surrogate like the $L_1$ penalty in Lasso. Exact $L_0$ regularization is normally avoided because it's intractable to search combinatorially ($2^N$ candidates, see "Why this is hard" above) and impossible to optimize by gradient descent — it isn't differentiable. Both obstacles vanish if the search operates over continuous vectors instead, which is exactly what PSO/GA/DE/SMA already do.

### Continuous relaxation

Replace the discrete indicator $s_i \in \{0, 1\}$ with a continuous weight $w_i \in [0, 1]$, and recover a discrete mask by thresholding at $0.5$:

$$\text{mask}(w)_i = \mathbb{1}[w_i > 0.5]$$

The optimizer minimizes the cost function evaluated on that mask:

$$f(w) = \underbrace{\Big(1 - \text{CV accuracy on features where } w_i > 0.5\Big)}_{\text{fidelity term}} \;+\; \underbrace{\alpha \cdot \frac{|\{i : w_i > 0.5\}|}{N}}_{\text{sparsity term}}$$

### Two terms, one trade-off

- **Fidelity term** $\big(1 - \text{CV accuracy}\big)$ turns "maximize accuracy" into "minimize error" to match every optimizer's minimize-only convention. As a function of $w$ it's piecewise-constant — it only changes when a weight crosses the $0.5$ threshold and flips a feature in or out of the mask — so it's discontinuous and has zero gradient almost everywhere.
- **Sparsity term** $\alpha \cdot |\text{mask}(w)|/N$ is a normalized, linear penalty on subset size: the true $L_0$ count, not Lasso's $L_1$ relaxation of it.
- Adding them together is a **scalarization** of what is really a two-objective problem (maximize accuracy, minimize feature count) into a single scalar: each choice of $\alpha$ picks one point on the accuracy-vs-sparsity Pareto front. $\alpha = 0$ removes any incentive to drop features (the optimizer keeps close to the full set, since extra features rarely hurt an SVM's CV accuracy much); larger $\alpha$ trades accuracy for a smaller subset. $\alpha = 0.02$ is used below — small enough that a feature is only dropped once it stops earning its share of accuracy.

Because $f$ is discontinuous and effectively derivative-free — it depends on $w$ only through the thresholded mask — no gradient-based method could optimize it directly. That's precisely the class of problem population-based metaheuristics are suited for: they only ever need to *evaluate* $f$ at a point, never differentiate it.

In [ ]:
def feature_subset_error(weights):
    mask = weights > 0.5
    if not mask.any():
        return 1.0
    score = cross_val_score(SVC(), X[:, mask], y, cv=5).mean()
    return (1 - score) + ALPHA * mask.sum() / NUM_FEATURES

bounds = [(0.0, 1.0)] * NUM_FEATURES

np.random.seed(0)
result = ParticleSwarmOptimization(num_particles=15, max_iterations=15).optimize(feature_subset_error, bounds)
mask = result.best_solution > 0.5

baseline_accuracy = cross_val_score(SVC(), X, y, cv=5).mean()
selected_accuracy = cross_val_score(SVC(), X[:, mask], y, cv=5).mean()

print(f'Selected {mask.sum()}/{NUM_FEATURES} features')
print(f'CV accuracy, all features     : {baseline_accuracy:.4f}')
print(f'CV accuracy, selected features: {selected_accuracy:.4f}')

Selected 9/30 features
CV accuracy, all features     : 0.9122
CV accuracy, selected features: 0.9385


In [4]:
import plotly.graph_objects as go

# Sweep one feature's weight through [0, 1], holding the rest at a fixed random
# baseline, to make the threshold discontinuity in f(w) concrete: it jumps
# whenever w_0 crosses 0.5 and the mask gains or drops that feature.
rng = np.random.default_rng(0)
base_weights = rng.random(NUM_FEATURES)
sweep_values = np.linspace(0.0, 1.0, 41)

costs = []
for v in sweep_values:
    w = base_weights.copy()
    w[0] = v
    costs.append(feature_subset_error(w))

fig = go.Figure(go.Scatter(x=sweep_values, y=costs, mode='lines+markers'))
fig.add_vline(x=0.5, line_dash='dash', line_color='gray', annotation_text='threshold')
fig.update_layout(
    title='f(w) is discontinuous at w_0 = 0.5 (rest of the weights held fixed)',
    xaxis_title='w_0',
    yaxis_title='f(w)',
)
fig.show()

In [5]:
fig = go.Figure(go.Scatter(y=result.fitness_history, mode='lines+markers'))
fig.update_layout(
    title='PSO convergence on the feature-selection objective',
    xaxis_title='iteration',
    yaxis_title='best fitness (1 - CV accuracy + penalty)',
)
fig.show()

In [6]:
feature_names = load_breast_cancer().feature_names
order = np.argsort(result.best_solution)[::-1]

fig = go.Figure(go.Bar(
    x=result.best_solution[order],
    y=feature_names[order],
    orientation='h',
    marker_color=['#2ca02c' if m else '#d62728' for m in mask[order]],
))
fig.add_vline(x=0.5, line_dash='dash', line_color='gray', annotation_text='threshold')
fig.update_layout(
    title='Feature weights and selection threshold (green = selected)',
    xaxis_title='weight w_i',
    yaxis_title='feature',
    height=700,
)
fig.show()

## Comparing metaheuristics on the same problem

The continuous-relaxation trick isn't specific to PSO — `GeneticAlgorithm`, `DifferentialEvolution`, and `SlimeMouldAlgorithm` share the same `optimize(objective_fn, bounds)` interface, so they can be dropped in unmodified. All four are run here on the identical objective with a matched, modest budget (15 population/particles x 15 iterations/generations) to compare how they trade off accuracy against sparsity.

In [7]:
import pandas as pd

from metaheuristics.algorithms.differential_evolution import DifferentialEvolution
from metaheuristics.algorithms.genetic_algorithm import GeneticAlgorithm
from metaheuristics.algorithms.slime_mould import SlimeMouldAlgorithm

POP, ITER = 15, 15

algorithms = {
    'GA': GeneticAlgorithm(population_size=POP, max_generations=ITER),
    'DE': DifferentialEvolution(population_size=POP, max_generations=ITER),
    'SMA': SlimeMouldAlgorithm(population_size=POP, max_iterations=ITER),
    'PSO': ParticleSwarmOptimization(num_particles=POP, max_iterations=ITER),
}

rows, curves = [], {}
for name, algo in algorithms.items():
    np.random.seed(0)
    res = algo.optimize(feature_subset_error, bounds)
    algo_mask = res.best_solution > 0.5
    accuracy = cross_val_score(SVC(), X[:, algo_mask], y, cv=5).mean() if algo_mask.any() else float('nan')
    rows.append({
        'algorithm': name,
        'num_features': int(algo_mask.sum()),
        'cv_accuracy': accuracy,
        'best_fitness': res.best_fitness,
    })
    curves[name] = res.fitness_history

comparison_df = pd.DataFrame(rows).set_index('algorithm')
comparison_df

,num_features,cv_accuracy,best_fitness
algorithm,,,
GA,9,0.938519,0.067481
DE,9,0.938519,0.067481
SMA,3,0.940289,0.061711
PSO,9,0.938519,0.067481


In [8]:
fig = go.Figure()
for name, history in curves.items():
    fig.add_trace(go.Scatter(y=history, name=name))
fig.update_layout(
    title='Convergence comparison across algorithms',
    xaxis_title='iteration',
    yaxis_title='best fitness',
)
fig.show()

In [9]:
accuracy_fig = go.Figure(go.Bar(
    x=['baseline (all features)'] + list(comparison_df.index),
    y=[baseline_accuracy] + list(comparison_df['cv_accuracy']),
))
accuracy_fig.update_layout(
    title='CV accuracy: baseline vs. each algorithm\'s selected subset',
    yaxis_title='CV accuracy',
    yaxis_range=[0.85, 1.0],
)
accuracy_fig.show()

count_fig = go.Figure(go.Bar(x=comparison_df.index, y=comparison_df['num_features']))
count_fig.add_hline(y=NUM_FEATURES, line_dash='dash', line_color='gray', annotation_text='all features')
count_fig.update_layout(
    title='Number of features selected per algorithm',
    yaxis_title='features selected',
)
count_fig.show()

## Next steps

Wrapping this continuous-relaxation approach as a scikit-learn-compatible selector (`BaseEstimator` + `SelectorMixin`, `fit`/`transform`) is planned future work, so it can be benchmarked directly — under a common API — against scikit-learn's own feature-selection methods such as `RFE`, `SelectKBest`, and `SelectFromModel`.